# LushProtein — CEO Retention Strategy: Data Derivation & Backup

**Purpose:** Step-by-step derivation of every number used in the retention strategy proposal.
All figures are computed directly from the `outputs_finals/` data — nothing is hardcoded.

**Source notebooks referenced:**
- `EDA/lushprotein_decile.ipynb` — Decile assignments (profit D1–D5, order freq D1–D5)
- `EDA/lushprotein_decile_time-based.ipynb` — Time-based customer behaviour (recency, repeat funnel, cadence)

**Scope:** Only D1 and D2 profit-decile customers (D3–D5 excluded — too low ROI for retention spend).

**Tier definitions:**
| Tier | Criteria | Rationale |
|---|---|---|
| **Tier 1** | Profit D1 AND Freq D1 | Highest value AND highest frequency — true VIPs |
| **Tier 2** | Profit D1 AND Freq ≠ D1 | High-value whales — big baskets, infrequent |
| **Tier 3** | Profit D2 AND Freq D1 | Frequent loyals — lower basket but strong habit |
| **Tier 4** | Profit D2 AND Freq ≠ D1 | Rising stars — one more purchase away from Tier 2/3 |

---

**Outputs generated:**
1. `outputs_finals/tier_customer_map.csv` — Customer ID → tier mapping for all calculations
2. Charts for each strategic section

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

BASE_DIR   = Path(".").resolve().parent          # EDA/ → lushprotein/
FINALS_DIR = BASE_DIR / "EDA" / "outputs_finals"
PROPOSAL_DIR = FINALS_DIR / "proposal"
PROPOSAL_DIR.mkdir(parents=True, exist_ok=True)  # create if it doesn't exist

# ── Retention budget parameters (tunable) ──────────────────────────────────────
RETENTION_BUDGET_PCT = {
    "Tier 1": 0.15,  # 15% of that tier's avg GP/customer
    "Tier 2": 0.12,
    "Tier 3": 0.12,
    "Tier 4": 0.10,
}

# ── Churn thresholds (consistent with lushprotein_decile_time-based.ipynb) ─────
CHURN_DAYS     = 180
AT_RISK_DAYS   = 90
REFERENCE_DATE = pd.Timestamp("2026-03-30", tz="UTC")

TIER_ORDER = ["Tier 1", "Tier 2", "Tier 3", "Tier 4"]
TIER_COLORS = {
    "Tier 1": "#1D3557",
    "Tier 2": "#457B9D",
    "Tier 3": "#2A9D8F",
    "Tier 4": "#A8DADC",
}

print("Finals dir  :", FINALS_DIR)
print("Proposal dir:", PROPOSAL_DIR)
print("Source files:", sorted([f.name for f in FINALS_DIR.iterdir() if f.is_file()]))

## 1. Load Source Data

Two pre-computed files from the source notebooks:
- `decile_customer_table.csv` — decile assignments + GP, revenue, subscription, categories
- `decile_timebased_table.csv` — recency, days-to-2nd, lifespan per customer

**Do not re-derive deciles here** — we use the authoritative outputs from `lushprotein_decile.ipynb`
to avoid any drift. If you need to re-run deciles, run that notebook first.

In [44]:
# ── Decile assignments (from lushprotein_decile.ipynb) ─────────────────────────
decile_df = pd.read_csv(
    FINALS_DIR / "decile_customer_table.csv",
    dtype={"customer_id": str}
)

# ── Time-based metrics (from lushprotein_decile_time-based.ipynb) ──────────────
timebased_df = pd.read_csv(
    FINALS_DIR / "decile_timebased_table.csv",
    dtype={"customer_id": str}
)

print(f"Decile table    : {len(decile_df):,} customers")
print(f"Time-based table: {len(timebased_df):,} customers")
print()
print("Decile table columns   :", list(decile_df.columns))
print("Time-based table columns:", list(timebased_df.columns))

Decile table    : 4,290 customers
Time-based table: 4,290 customers

Decile table columns   : ['customer_id', 'finals_orders', 'finals_revenue', 'true_gross_profit', 'avg_margin_pct', 'cogs_coverage_pct', 'profit_decile', 'order_freq_decile', 'first_channel', 'first_product_cat', 'first_store', 'ever_subscribed', 'cohort_month', 'acq_year', 'n_categories_ever', 'is_top_profit', 'is_top_freq', 'is_top_both']
Time-based table columns: ['customer_id', 'order_freq_decile', 'finals_orders', 'days_to_second', 'days_to_third', 'days_2nd_to_3rd', 'lifespan_days', 'recency_days', 'recency_status', 'first_order_date', 'last_order_date']


In [45]:
# ── Merge on customer_id ───────────────────────────────────────────────────────
# Use left join on decile_df — it is the authoritative pool (4,290 customers).
# The time-based table covers the same pool; inner join would be equivalent.
time_cols_to_join = [
    "customer_id",
    "days_to_second", "days_to_third", "days_2nd_to_3rd",
    "lifespan_days", "recency_days", "recency_status",
]
available_time_cols = [c for c in time_cols_to_join if c in timebased_df.columns]

master = decile_df.merge(
    timebased_df[available_time_cols],
    on="customer_id",
    how="left"
)

# Normalise decile columns to strings
master["profit_decile"]     = master["profit_decile"].astype(str)
master["order_freq_decile"] = master["order_freq_decile"].astype(str)

print(f"Master table: {len(master):,} customers")
print()
print("Profit decile counts:")
print(master["profit_decile"].value_counts().sort_index())
print()
print("Order freq decile counts:")
print(master["order_freq_decile"].value_counts().sort_index())

Master table: 4,290 customers

Profit decile counts:
profit_decile
D1    858
D2    858
D3    858
D4    858
D5    858
Name: count, dtype: int64

Order freq decile counts:
order_freq_decile
D1    858
D2    858
D3    858
D4    858
D5    858
Name: count, dtype: int64


## 2. Tier Assignment Logic

We assign each customer to exactly one tier using a **priority waterfall**:
1. **Tier 1** — Profit D1 AND Freq D1 (highest GP + highest frequency)
2. **Tier 2** — Profit D1 AND Freq ≠ D1 (profit whales, infrequent buyers)
3. **Tier 3** — Profit D2 AND Freq D1 (frequent loyals, moderate basket)
4. **Tier 4** — Profit D2 AND Freq ≠ D1 (rising stars, not yet habitual)
5. **Excluded** — All Profit D3, D4, D5 customers (too low ROI for retention spend)

Why this ordering? Tier 1 takes priority because a Profit D1 + Freq D1 customer is more
valuable than a Profit D1 + Freq D2 customer. Tier 3 captures the subset of D2-profit customers
who have already shown strong purchase frequency — they are functionally more loyal than a
D1-profit customer who bought once with a big basket.

In [46]:
def assign_tier(row):
    p = row["profit_decile"]
    f = row["order_freq_decile"]
    if p == "D1" and f == "D1":
        return "Tier 1"
    elif p == "D1" and f != "D1":
        return "Tier 2"
    elif p == "D2" and f == "D1":
        return "Tier 3"
    elif p == "D2" and f != "D1":
        return "Tier 4"
    else:
        return "Excluded"  # D3, D4, D5 profit

master["tier"] = master.apply(assign_tier, axis=1)

tier_counts = master["tier"].value_counts()
print("Tier assignment results:")
for tier in TIER_ORDER + ["Excluded"]:
    n = tier_counts.get(tier, 0)
    print(f"  {tier:<12}: {n:>5,} customers")
print(f"  {'Total':<12}: {len(master):>5,} customers")
print()

# Sanity check: no customer appears in multiple tiers
active_tiers = master[master["tier"] != "Excluded"]
assert active_tiers["customer_id"].nunique() == len(active_tiers), "Duplicate assignments!"
print(f"Active customers (D1 + D2 profit only): {len(active_tiers):,}")
print(f"Excluded (D3-D5 profit): {(master['tier'] == 'Excluded').sum():,}")

Tier assignment results:
  Tier 1      :   500 customers
  Tier 2      :   358 customers
  Tier 3      :   195 customers
  Tier 4      :   663 customers
  Excluded    : 2,574 customers
  Total       : 4,290 customers

Active customers (D1 + D2 profit only): 1,716
Excluded (D3-D5 profit): 2,574


## 3. Export: Customer Tier Map CSV

The primary output of this notebook — a flat CSV mapping each in-scope customer to their tier,
alongside all financial and behavioural fields needed for further calculations.

In [ ]:
export_cols = [
    "customer_id",
    "tier",
    "profit_decile",
    "order_freq_decile",
    "finals_orders",
    "finals_revenue",
    "true_gross_profit",
    "avg_margin_pct",
    "cogs_coverage_pct",
    "ever_subscribed",
    "n_categories_ever",
    "first_channel",
    "first_product_cat",
    "acq_year",
    "days_to_second",
    "lifespan_days",
    "recency_days",
    "recency_status",
]
available_cols = [c for c in export_cols if c in master.columns]

tier_map = (
    master[master["tier"] != "Excluded"][available_cols]
    .sort_values(["tier", "true_gross_profit"], ascending=[True, False])
    .reset_index(drop=True)
)

out_path = PROPOSAL_DIR / "tier_customer_map.csv"
tier_map.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows : {len(tier_map):,}")
print()
print("Counts per tier:")
print(tier_map["tier"].value_counts())
print()
print("Preview (first 5 rows):")
display(tier_map.head())

## 4. Tier Profiles — Who Are They?

For each tier we compute the core metrics that drive the retention strategy:
- **n_customers** — pool size
- **total_gp / avg_gp** — absolute and per-customer gross profit
- **avg_revenue / avg_aov** — revenue and average order value
- **avg_orders** — purchase frequency
- **pct_subscribed** — share already on subscription (retention lock-in proxy)
- **avg_categories** — category breadth (higher = more engaged)

These are the numbers the strategy references when calculating budget and expected return.

In [48]:
df_active = tier_map.copy()

profiles = []
for tier in TIER_ORDER:
    g = df_active[df_active["tier"] == tier]
    n = len(g)
    row = {
        "tier":             tier,
        "n_customers":      n,
        "total_gp":         g["true_gross_profit"].sum(),
        "avg_gp":           g["true_gross_profit"].mean(),
        "median_gp":        g["true_gross_profit"].median(),
        "avg_revenue":      g["finals_revenue"].mean(),
        "avg_orders":       g["finals_orders"].mean(),
        "pct_subscribed":   g["ever_subscribed"].mean() if "ever_subscribed" in g.columns else np.nan,
        "avg_categories":   g["n_categories_ever"].mean() if "n_categories_ever" in g.columns else np.nan,
        "avg_margin_pct":   g["avg_margin_pct"].mean(),
    }
    # AOV = avg revenue per order
    row["avg_aov"] = row["avg_revenue"] / row["avg_orders"] if row["avg_orders"] > 0 else np.nan
    profiles.append(row)

prof_df = pd.DataFrame(profiles)

total_pool_gp = df_active["true_gross_profit"].sum()
prof_df["pct_of_active_gp"] = prof_df["total_gp"] / total_pool_gp

fmt = {
    "total_gp":         lambda x: f"S${x:,.0f}",
    "avg_gp":           lambda x: f"S${x:,.2f}",
    "median_gp":        lambda x: f"S${x:,.2f}",
    "avg_revenue":      lambda x: f"S${x:,.2f}",
    "avg_aov":          lambda x: f"S${x:,.2f}",
    "avg_orders":       lambda x: f"{x:.1f}",
    "pct_subscribed":   lambda x: f"{x:.1%}",
    "avg_categories":   lambda x: f"{x:.1f}",
    "avg_margin_pct":   lambda x: f"{x:.1f}%",
    "pct_of_active_gp": lambda x: f"{x:.1%}",
}

display_cols = ["tier", "n_customers", "total_gp", "avg_gp", "median_gp",
                "avg_revenue", "avg_aov", "avg_orders",
                "pct_subscribed", "avg_categories", "pct_of_active_gp"]

print(f"TIER PROFILES  (total active-pool GP = S${total_pool_gp:,.0f})")
print()
print(prof_df[display_cols].to_string(index=False, formatters=fmt))

TIER PROFILES  (total active-pool GP = S$267,094)

  tier  n_customers  total_gp   avg_gp median_gp avg_revenue  avg_aov avg_orders pct_subscribed avg_categories pct_of_active_gp
Tier 1          500 S$132,691 S$265.38  S$188.88    S$430.40  S$99.68        4.3          51.0%            2.5            49.7%
Tier 2          358  S$64,691 S$180.70  S$141.59    S$296.89 S$216.91        1.4           8.9%            1.9            24.2%
Tier 3          195  S$16,163  S$82.89   S$84.33    S$143.28  S$51.08        2.8          52.3%            2.0             6.1%
Tier 4          663  S$53,548  S$80.77   S$80.45    S$115.79 S$106.18        1.1          13.4%            1.6            20.0%


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("Tier Profiles — D1 & D2 Profit Customers Only",
             fontsize=13, fontweight="bold")

colors = [TIER_COLORS[t] for t in TIER_ORDER]

# Avg GP
ax = axes[0]
bars = ax.bar(TIER_ORDER, prof_df["avg_gp"], color=colors, edgecolor="white")
ax.set_title("Avg GP per Customer (S$)")
ax.set_ylabel("SGD")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"S${x:,.0f}"))
for bar, val in zip(bars, prof_df["avg_gp"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f"S${val:,.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# Avg Orders
ax = axes[1]
bars = ax.bar(TIER_ORDER, prof_df["avg_orders"], color=colors, edgecolor="white")
ax.set_title("Avg Orders per Customer")
ax.set_ylabel("Orders")
for bar, val in zip(bars, prof_df["avg_orders"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# % Subscribed
ax = axes[2]
bars = ax.bar(TIER_ORDER, prof_df["pct_subscribed"] * 100, color=colors, edgecolor="white")
ax.set_title("% Ever Subscribed")
ax.set_ylabel("%")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
for bar, val in zip(bars, prof_df["pct_subscribed"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1%}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# % of Total GP
ax = axes[3]
bars = ax.bar(TIER_ORDER, prof_df["pct_of_active_gp"] * 100, color=colors, edgecolor="white")
ax.set_title("% of Active-Pool Total GP")
ax.set_ylabel("%")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
for bar, val in zip(bars, prof_df["pct_of_active_gp"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1%}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(PROPOSAL_DIR / "proposal_tier_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: proposal_tier_profiles.png")

## 5. Recency & Churn Status by Tier

Before proposing any retention spend, we need to know how much of each tier is already churned.

Thresholds (same as `lushprotein_decile_time-based.ipynb`):
- **Active**: `recency_days < 90` — purchased within 3 months
- **At-risk**: `90 ≤ recency_days < 180` — no purchase in 3–6 months
- **Likely churned**: `recency_days ≥ 180` — silent for 6+ months

This drives two separate strategies:
- **Active + At-risk customers** → proactive retention (protect before they leave)
- **Likely churned customers** → win-back campaign (bring them back)

In [50]:
def recency_label(days):
    if pd.isna(days):
        return "Unknown"
    elif days < AT_RISK_DAYS:
        return "Active"
    elif days < CHURN_DAYS:
        return "At-risk"
    else:
        return "Likely churned"

# Re-derive from recency_days to ensure consistency (don't rely on pre-computed label)
df_active["recency_bucket"] = df_active["recency_days"].apply(recency_label)

recency_rows = []
for tier in TIER_ORDER:
    g = df_active[df_active["tier"] == tier]
    n = len(g)
    counts = g["recency_bucket"].value_counts()
    median_rec = g["recency_days"].median()
    recency_rows.append({
        "tier":              tier,
        "n_customers":       n,
        "median_recency_days": median_rec,
        "n_active":          counts.get("Active", 0),
        "n_at_risk":         counts.get("At-risk", 0),
        "n_churned":         counts.get("Likely churned", 0),
        "pct_active":        counts.get("Active", 0) / n,
        "pct_at_risk":       counts.get("At-risk", 0) / n,
        "pct_churned":       counts.get("Likely churned", 0) / n,
    })

rec_df = pd.DataFrame(recency_rows)

fmt_rec = {
    "median_recency_days": lambda x: f"{x:.0f}d",
    "pct_active":          lambda x: f"{x:.1%}",
    "pct_at_risk":         lambda x: f"{x:.1%}",
    "pct_churned":         lambda x: f"{x:.1%}",
}

print(f"RECENCY STATUS BY TIER  "
      f"(Active <{AT_RISK_DAYS}d | At-risk {AT_RISK_DAYS}-{CHURN_DAYS}d | Churned >={CHURN_DAYS}d)")
print()
print(rec_df.to_string(index=False, formatters=fmt_rec))
print()

# Headline numbers
for tier in TIER_ORDER:
    r = rec_df[rec_df["tier"] == tier].iloc[0]
    print(f"{tier}: {r['n_churned']:.0f} churned ({r['pct_churned']:.1%}), "
          f"{r['n_active']:.0f} active ({r['pct_active']:.1%}), "
          f"median recency {r['median_recency_days']:.0f} days")

RECENCY STATUS BY TIER  (Active <90d | At-risk 90-180d | Churned >=180d)

  tier  n_customers median_recency_days  n_active  n_at_risk  n_churned pct_active pct_at_risk pct_churned
Tier 1          500                308d       104         38        358      20.8%        7.6%       71.6%
Tier 2          358                606d        39         10        309      10.9%        2.8%       86.3%
Tier 3          195                414d        31          5        159      15.9%        2.6%       81.5%
Tier 4          663                440d        99         43        521      14.9%        6.5%       78.6%

Tier 1: 358 churned (71.6%), 104 active (20.8%), median recency 308 days
Tier 2: 309 churned (86.3%), 39 active (10.9%), median recency 606 days
Tier 3: 159 churned (81.5%), 31 active (15.9%), median recency 414 days
Tier 4: 521 churned (78.6%), 99 active (14.9%), median recency 440 days


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle(f"Recency Status by Tier  "
             f"(Active <{AT_RISK_DAYS}d | At-risk {AT_RISK_DAYS}–{CHURN_DAYS}d | Churned >={CHURN_DAYS}d)",
             fontsize=12, fontweight="bold")

status_colors = {"Active": "#2A9D8F", "At-risk": "#E9C46A", "Likely churned": "#E76F51"}

# Stacked bar — % breakdown
ax = axes[0]
bottoms = np.zeros(len(TIER_ORDER))
for status, col in [("Active", "pct_active"), ("At-risk", "pct_at_risk"), ("Likely churned", "pct_churned")]:
    vals = rec_df[col].values * 100
    bars = ax.bar(TIER_ORDER, vals, bottom=bottoms,
                  color=status_colors[status], edgecolor="white", label=status)
    for bar, v, b in zip(bars, vals, bottoms):
        if v > 5:
            ax.text(bar.get_x() + bar.get_width()/2, b + v/2,
                    f"{v:.0f}%", ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")
    bottoms += vals
ax.set_title("% by Recency Status")
ax.set_ylabel("% of Tier")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.legend(loc="lower right", fontsize=9)

# Absolute count of churned customers
ax = axes[1]
colors_list = [TIER_COLORS[t] for t in TIER_ORDER]
bars = ax.bar(TIER_ORDER, rec_df["n_churned"], color=colors_list, edgecolor="white")
ax.set_title("# Likely Churned Customers (>180 days)")
ax.set_ylabel("Customers")
for bar, val, pct in zip(bars, rec_df["n_churned"], rec_df["pct_churned"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:.0f}\n({pct:.1%})", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(PROPOSAL_DIR / "proposal_recency_by_tier.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: proposal_recency_by_tier.png")

## 6. Win-Back Opportunity — Churned Tier 1 & Tier 2

Churned customers who were formerly in our highest-value tiers represent the single biggest
short-term revenue recovery opportunity — zero acquisition cost, proven product-market fit.

**Approach:** Model three recovery scenarios per tier:
- **Conservative (5%)** — 5% of churned customers reactivate at 50% of prior avg GP
- **Base (10%)** — 10% reactivate at 50% of prior avg GP
- **Optimistic (10%)** — 10% reactivate and return to full avg GP

Win-back spend assumption: **S$30/customer** (personalised outreach, gift, or experience voucher).

In [52]:
WINBACK_COST_PER_PERSON = 30  # SGD
WINBACK_TIERS = ["Tier 1", "Tier 2", "Tier 3"]  # Tier 4 win-back ROI too low

print("WIN-BACK OPPORTUNITY — Churned Tier 1, Tier 2, Tier 3")
print(f"(Win-back outreach cost assumption: S${WINBACK_COST_PER_PERSON}/customer)")
print()
print(f"{'Tier':<10} {'Churned':>8} {'Avg GP':>10} {'Win-back budget':>16} "
      f"{'5% conv (50% GP)':>18} {'10% conv (50% GP)':>19} {'10% conv (full GP)':>20}")
print("-" * 100)

winback_rows = []
for tier in WINBACK_TIERS:
    r_rec  = rec_df[rec_df["tier"] == tier].iloc[0]
    r_prof = prof_df[prof_df["tier"] == tier].iloc[0]
    n_churned = r_rec["n_churned"]
    avg_gp    = r_prof["avg_gp"]
    budget    = n_churned * WINBACK_COST_PER_PERSON
    conv_5_half  = int(n_churned * 0.05) * (avg_gp * 0.5)
    conv_10_half = int(n_churned * 0.10) * (avg_gp * 0.5)
    conv_10_full = int(n_churned * 0.10) * avg_gp
    roi_base = conv_10_half / budget if budget > 0 else 0

    winback_rows.append({
        "tier": tier, "n_churned": n_churned, "avg_gp": avg_gp,
        "winback_budget": budget,
        "conv_5_half": conv_5_half, "conv_10_half": conv_10_half, "conv_10_full": conv_10_full,
        "roi_base": roi_base,
    })

    print(f"{tier:<10} {n_churned:>8,.0f} {avg_gp:>9,.0f}S$ {budget:>15,.0f}S$ "
          f"{conv_5_half:>17,.0f}S$ {conv_10_half:>18,.0f}S$ {conv_10_full:>19,.0f}S$")

wb_df = pd.DataFrame(winback_rows)
print("-" * 100)
total_budget  = wb_df["winback_budget"].sum()
total_base    = wb_df["conv_10_half"].sum()
total_optimistic = wb_df["conv_10_full"].sum()
print(f"{'TOTAL':<10} {wb_df['n_churned'].sum():>8,.0f} {'':>10} {total_budget:>15,.0f}S$ "
      f"{'':>18} {total_base:>18,.0f}S$ {total_optimistic:>19,.0f}S$")
print()
print(f"Base scenario net GP recovered (after win-back spend): S${total_base - total_budget:,.0f}")
print(f"Base scenario ROI: {total_base/total_budget:.1f}x")
print()
print("Note: Prioritise churned 180–365 days (more winnable) over 365+ days.")
print("      Segment the churned list by recency before sending outreach.")

WIN-BACK OPPORTUNITY — Churned Tier 1, Tier 2, Tier 3
(Win-back outreach cost assumption: S$30/customer)

Tier        Churned     Avg GP  Win-back budget   5% conv (50% GP)   10% conv (50% GP)   10% conv (full GP)
----------------------------------------------------------------------------------------------------
Tier 1          358       265S$          10,740S$             2,256S$              4,644S$               9,288S$
Tier 2          309       181S$           9,270S$             1,355S$              2,711S$               5,421S$
Tier 3          159        83S$           4,770S$               290S$                622S$               1,243S$
----------------------------------------------------------------------------------------------------
TOTAL           826                     24,780S$                                 7,976S$              15,953S$

Base scenario net GP recovered (after win-back spend): S$-16,804
Base scenario ROI: 0.3x

Note: Prioritise churned 180–365 days (more

In [53]:
# Segment churned Platinum & Gold by recency band — prioritise 180-365d over 365+d
churned_vip = df_active[
    (df_active["tier"].isin(WINBACK_TIERS)) &
    (df_active["recency_days"] >= CHURN_DAYS)
].copy()

churned_vip["winback_priority"] = churned_vip["recency_days"].apply(
    lambda d: "Priority (180-365d)" if d < 365 else "Lower (365d+)"
)

priority_breakdown = churned_vip.groupby(["tier", "winback_priority"]).agg(
    n_customers=("customer_id", "count"),
    avg_gp=("true_gross_profit", "mean"),
).reset_index()

print("WIN-BACK PRIORITISATION — by recency band")
print(priority_breakdown.to_string(index=False,
    formatters={"avg_gp": lambda x: f"S${x:,.0f}"}))
print()
priority_count = churned_vip[churned_vip["winback_priority"] == "Priority (180-365d)"]["customer_id"].count()
print(f"→ Focus win-back spend on {priority_count:,} customers (180-365d inactive).")
print(f"  Budget for priority segment: {priority_count} × S${WINBACK_COST_PER_PERSON} = "
      f"S${priority_count * WINBACK_COST_PER_PERSON:,}")

WIN-BACK PRIORITISATION — by recency band
  tier    winback_priority  n_customers avg_gp
Tier 1       Lower (365d+)          206  S$277
Tier 1 Priority (180-365d)          152  S$230
Tier 2       Lower (365d+)          237  S$202
Tier 2 Priority (180-365d)           72  S$136
Tier 3       Lower (365d+)          113   S$83
Tier 3 Priority (180-365d)           46   S$84

→ Focus win-back spend on 270 customers (180-365d inactive).
  Budget for priority segment: 270 × S$30 = S$8,100


## 7. Subscription Conversion — Tier 3 Non-Subscribers

Tier 3 customers (Profit D2 + Freq D1) already order regularly but many haven't locked into a
subscription. This is the highest-confidence subscription conversion opportunity:
they've **proved frequency without a subscription** — any friction to convert is purely mechanical.

Locking them into a subscription turns uncertain reorders into guaranteed revenue.

In [54]:
tier3 = df_active[df_active["tier"] == "Tier 3"].copy()

n_tier3_total    = len(tier3)
n_subscribed     = tier3["ever_subscribed"].sum() if "ever_subscribed" in tier3.columns else 0
n_not_subscribed = n_tier3_total - n_subscribed
pct_subscribed   = n_subscribed / n_tier3_total if n_tier3_total > 0 else 0

avg_gp_tier3 = tier3["true_gross_profit"].mean()

print("TIER 3 — SUBSCRIPTION STATUS")
print(f"  Total Tier 3 customers : {n_tier3_total:,}")
print(f"  Already subscribed     : {n_subscribed:,} ({pct_subscribed:.1%})")
print(f"  NOT yet subscribed     : {n_not_subscribed:,} ({1-pct_subscribed:.1%})  ← target for conversion")
print(f"  Avg GP (Tier 3)        : S${avg_gp_tier3:,.2f}/customer/year")
print()

# Subscription conversion scenarios
print("SUBSCRIPTION CONVERSION SCENARIOS (non-subscribed Tier 3)")
print(f"{'Scenario':<25} {'Converts':>10} {'Locked-in GP/yr':>18} {'vs uncertain reorder':>22}")
print("-" * 78)
for pct_convert, label in [(0.30, "Conservative (30%)"), (0.50, "Base (50%)"), (0.70, "Optimistic (70%)")]:
    n_converts = int(n_not_subscribed * pct_convert)
    locked_gp  = n_converts * avg_gp_tier3
    print(f"{label:<25} {n_converts:>10,} {locked_gp:>17,.0f}S$ "
          f"  (was at-risk churn)")
print()
print("Key insight: Tier 3 customers already buy at Freq D1 cadence (~44 days).")
print("Subscription converts habit into contract — same GP, lower churn risk.")
print()

# Natural reorder cadence for Tier 3 — informs subscription interval setting
tier3_repeat = tier3[tier3["days_to_second"].notna()]
if len(tier3_repeat) > 0:
    median_cadence = tier3_repeat["days_to_second"].median()
    p25_cadence    = tier3_repeat["days_to_second"].quantile(0.25)
    p75_cadence    = tier3_repeat["days_to_second"].quantile(0.75)
    print(f"Tier 3 natural reorder cadence (1st→2nd purchase):")
    print(f"  Median: {median_cadence:.0f} days | IQR: {p25_cadence:.0f}–{p75_cadence:.0f} days")
    print(f"  → Recommended subscription interval: {int(round(median_cadence/5)*5)} days")

TIER 3 — SUBSCRIPTION STATUS
  Total Tier 3 customers : 195
  Already subscribed     : 102 (52.3%)
  NOT yet subscribed     : 93 (47.7%)  ← target for conversion
  Avg GP (Tier 3)        : S$82.89/customer/year

SUBSCRIPTION CONVERSION SCENARIOS (non-subscribed Tier 3)
Scenario                    Converts    Locked-in GP/yr   vs uncertain reorder
------------------------------------------------------------------------------
Conservative (30%)                27             2,238S$   (was at-risk churn)
Base (50%)                        46             3,813S$   (was at-risk churn)
Optimistic (70%)                  65             5,388S$   (was at-risk churn)

Key insight: Tier 3 customers already buy at Freq D1 cadence (~44 days).
Subscription converts habit into contract — same GP, lower churn risk.

Tier 3 natural reorder cadence (1st→2nd purchase):
  Median: 50 days | IQR: 28–120 days
  → Recommended subscription interval: 50 days


## 8. Tier 4 → Tier 2/3 Upgrade Funnel

Tier 4 customers (Profit D2, non-frequent) are the rising stars. A second purchase at a
higher basket value — or a cross-sell to a second category — could push them into Tier 2 or Tier 3.

Two levers:
1. **Timing the 2nd-purchase nudge** — reach them before their natural 2nd-purchase window closes
2. **Cross-sell to a second category** — customers with 2+ categories have significantly higher GP

In [55]:
tier4 = df_active[df_active["tier"] == "Tier 4"].copy()
n_tier4 = len(tier4)

# ── 2nd purchase timing ──────────────────────────────────────────────────────
tier4_with_2nd = tier4[tier4["days_to_second"].notna()]
tier4_one_time = tier4[tier4["days_to_second"].isna()]

pct_repeat = len(tier4_with_2nd) / n_tier4
print("TIER 4 — REPEAT PURCHASE STATUS")
print(f"  Total Tier 4 customers : {n_tier4:,}")
print(f"  Already made 2nd order : {len(tier4_with_2nd):,} ({pct_repeat:.1%})")
print(f"  One-time only          : {len(tier4_one_time):,} ({1-pct_repeat:.1%})  ← 2nd purchase target")

if len(tier4_with_2nd) > 0:
    med_2nd = tier4_with_2nd["days_to_second"].median()
    p25_2nd = tier4_with_2nd["days_to_second"].quantile(0.25)
    p75_2nd = tier4_with_2nd["days_to_second"].quantile(0.75)
    print(f"\nAmong those who returned — time to 2nd purchase:")
    print(f"  Median: {med_2nd:.0f} days | IQR: {p25_2nd:.0f}–{p75_2nd:.0f} days")
    print(f"  → Optimal intervention window: days {int(p25_2nd)}–{int(med_2nd)} after 1st order")

print()

# ── Category breadth impact on GP ────────────────────────────────────────────
print("TIER 4 — GP BY CATEGORY COUNT (breadth vs value)")
cat_gp = (
    tier4.groupby("n_categories_ever")["true_gross_profit"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_gp", "count": "n_customers"})
    .reset_index()
)
print(cat_gp.to_string(index=False,
    formatters={"avg_gp": lambda x: f"S${x:,.0f}"}))

print()

# ── Upgrade scenario math ─────────────────────────────────────────────────────
avg_gp_tier4     = tier4["true_gross_profit"].mean()
avg_gp_tier2_3   = df_active[df_active["tier"].isin(["Tier 2", "Tier 3"])]["true_gross_profit"].mean()
gp_uplift_per_customer = avg_gp_tier2_3 - avg_gp_tier4

print("TIER 4 → TIER 2/3 UPGRADE SCENARIOS")
print(f"  Avg GP (Tier 4)      : S${avg_gp_tier4:,.0f}")
print(f"  Avg GP (Tier 2 + 3)  : S${avg_gp_tier2_3:,.0f}")
print(f"  GP uplift if upgraded: S${gp_uplift_per_customer:,.0f} per customer")
print()
print(f"{'Scenario':<22} {'Upgrades':>10} {'Incremental GP':>17}")
print("-" * 52)
for pct, label in [(0.05, "Conservative (5%)"), (0.10, "Base (10%)"), (0.15, "Optimistic (15%)")]:
    n_up   = int(n_tier4 * pct)
    inc_gp = n_up * gp_uplift_per_customer
    print(f"{label:<22} {n_up:>10,} {inc_gp:>16,.0f}S$")

TIER 4 — REPEAT PURCHASE STATUS
  Total Tier 4 customers : 663
  Already made 2nd order : 60 (9.0%)
  One-time only          : 603 (91.0%)  ← 2nd purchase target

Among those who returned — time to 2nd purchase:
  Median: 47 days | IQR: 17–130 days
  → Optimal intervention window: days 16–47 after 1st order

TIER 4 — GP BY CATEGORY COUNT (breadth vs value)
 n_categories_ever avg_gp  n_customers
                 1   S$82          356
                 2   S$77          227
                 3   S$84           72
                 4   S$82            8

TIER 4 → TIER 2/3 UPGRADE SCENARIOS
  Avg GP (Tier 4)      : S$81
  Avg GP (Tier 2 + 3)  : S$146
  GP uplift if upgraded: S$65 per customer

Scenario                 Upgrades    Incremental GP
----------------------------------------------------
Conservative (5%)              33            2,160S$
Base (10%)                     66            4,319S$
Optimistic (15%)               99            6,479S$


## 9. Time-Based Behaviour by Tier

Derived from `lushprotein_decile_time-based.ipynb`. We re-compute by tier rather than by
decile to get the exact figures used in the strategy proposal.

In [ ]:
print("REPEAT PURCHASE FUNNEL BY TIER")
print(f"({'% of tier with at least N orders'})")
print()

funnel_rows = []
for tier in TIER_ORDER:
    g = df_active[df_active["tier"] == tier]
    n = len(g)
    row = {"tier": tier, "n_customers": n}
    for k in [2, 3, 4, 5]:
        row[f"pct_{k}plus"] = (g["finals_orders"] >= k).sum() / n
    funnel_rows.append(row)

funnel_df = pd.DataFrame(funnel_rows)

fmt_f = {f"pct_{k}plus": "{:.1%}".format for k in [2,3,4,5]}
print(funnel_df.to_string(index=False, formatters=fmt_f))
print()

# Chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Repeat Purchase Funnel by Tier", fontsize=13, fontweight="bold")

x = np.arange(len(TIER_ORDER))
width = 0.18
cols_plot = [f"pct_{k}plus" for k in [2,3,4,5]]
labels    = [">=2 orders", ">=3 orders", ">=4 orders", ">=5 orders"]
bar_colors = ["#1D3557", "#457B9D", "#2A9D8F", "#A8DADC"]

for i, (col, label, bc) in enumerate(zip(cols_plot, labels, bar_colors)):
    ax.bar(x + i * width, funnel_df[col] * 100, width, label=label, color=bc, edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(TIER_ORDER)
ax.set_ylabel("% of Tier")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(PROPOSAL_DIR / "proposal_repeat_funnel_by_tier.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: proposal_repeat_funnel_by_tier.png")

In [ ]:
# ── Time to 2nd purchase (among customers who made a 2nd purchase) ────────────
print("TIME TO 2nd PURCHASE — by Tier (repeat buyers only)")
print("(This is the intervention window: reach them before they go cold)")
print()

t2nd_rows = []
for tier in TIER_ORDER:
    g = df_active[(df_active["tier"] == tier) & (df_active["days_to_second"].notna())]
    n = len(g)
    if n == 0:
        t2nd_rows.append({"tier": tier, "n_eligible": 0, "median_days": np.nan,
                           "p25_days": np.nan, "p75_days": np.nan})
        continue
    t2nd_rows.append({
        "tier":        tier,
        "n_eligible":  n,
        "median_days": g["days_to_second"].median(),
        "p25_days":    g["days_to_second"].quantile(0.25),
        "p75_days":    g["days_to_second"].quantile(0.75),
    })

t2nd_df = pd.DataFrame(t2nd_rows)
print(t2nd_df.to_string(index=False,
    formatters={c: lambda x: f"{x:.0f}d" for c in ["median_days","p25_days","p75_days"]}))
print()

# Chart
fig, ax = plt.subplots(figsize=(9, 5))
ax.set_title("Days to 2nd Purchase — Median + IQR by Tier (repeat buyers only)",
             fontsize=12, fontweight="bold")

colors_list = [TIER_COLORS[t] for t in TIER_ORDER]
bars = ax.bar(TIER_ORDER, t2nd_df["median_days"], color=colors_list, edgecolor="white")

yerr_low  = t2nd_df["median_days"] - t2nd_df["p25_days"]
yerr_high = t2nd_df["p75_days"]   - t2nd_df["median_days"]
ax.errorbar(TIER_ORDER, t2nd_df["median_days"],
            yerr=[yerr_low, yerr_high],
            fmt="none", color="#333333", capsize=5, linewidth=1.5, label="IQR (p25-p75)")

ax.set_ylabel("Days")
ax.legend(fontsize=9)
for bar, val, n in zip(bars, t2nd_df["median_days"], t2nd_df["n_eligible"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:.0f}d\n(n={n:.0f})", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(PROPOSAL_DIR / "proposal_time_to_2nd_by_tier.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: proposal_time_to_2nd_by_tier.png")

In [ ]:
# ── Retention at 3 / 6 / 12 month milestones ─────────────────────────────────
# Reads source data from FINALS_DIR; writes chart to PROPOSAL_DIR
timebased_full = pd.read_csv(
    FINALS_DIR / "decile_timebased_table.csv",
    dtype={"customer_id": str},
    parse_dates=["first_order_date", "last_order_date"],
)

# Also need second_order_date from customers.parquet
cust_parquet = pd.read_parquet(FINALS_DIR / "customers.parquet")
cust_parquet["customer_id"] = cust_parquet["customer_id"].astype(str)

# Merge second_order_date into timebased_full
if "second_order_date" in cust_parquet.columns:
    timebased_full = timebased_full.merge(
        cust_parquet[["customer_id", "second_order_date"]],
        on="customer_id", how="left"
    )
    timebased_full["second_order_date"] = pd.to_datetime(timebased_full["second_order_date"], utc=True)

# Merge tier info
tier_info = df_active[["customer_id", "tier"]]
ret_base = timebased_full.merge(tier_info, on="customer_id", how="inner")

# Ensure datetime with tz
for col in ["first_order_date", "last_order_date"]:
    if col in ret_base.columns:
        ret_base[col] = pd.to_datetime(ret_base[col], utc=True)

milestones = [90, 180, 365]

def retention_milestone(df, tier_col, milestone_days, ref_date):
    if "second_order_date" not in df.columns:
        print("second_order_date not available — skipping milestone analysis")
        return pd.DataFrame()
    cutoff = ref_date - pd.Timedelta(days=milestone_days)
    eligible = df[df["first_order_date"] <= cutoff].copy()
    eligible["repeat_within"] = (
        eligible["second_order_date"].notna() &
        ((eligible["second_order_date"] - eligible["first_order_date"]).dt.total_seconds()
         / 86400 <= milestone_days)
    )
    rows = []
    for tier in TIER_ORDER:
        g = eligible[eligible[tier_col] == tier]
        n = len(g)
        pct = g["repeat_within"].mean() if n > 0 else np.nan
        rows.append({tier_col: tier, "n_eligible": n, f"pct_repeat_{milestone_days}d": pct})
    return pd.DataFrame(rows)

ret_merged = None
for m in milestones:
    r = retention_milestone(ret_base, "tier", m, REFERENCE_DATE)
    if r.empty:
        break
    ret_merged = r if ret_merged is None else ret_merged.merge(
        r[["tier", f"pct_repeat_{m}d"]], on="tier"
    )

if ret_merged is not None and not ret_merged.empty:
    fmt_ret = {f"pct_repeat_{m}d": "{:.1%}".format for m in milestones}
    print("RETENTION AT MILESTONES — by Tier")
    print("(% who placed a 2nd order within N days of 1st purchase)")
    print()
    print(ret_merged.to_string(index=False, formatters=fmt_ret))

    # Chart
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title("Retention at Milestones by Tier\n(% who placed 2nd order within N months)",
                 fontsize=12, fontweight="bold")
    x = np.arange(len(TIER_ORDER))
    width = 0.25
    m_labels = ["3 months (90d)", "6 months (180d)", "12 months (365d)"]
    m_colors = ["#1D3557", "#457B9D", "#A8DADC"]
    for i, (m, ml, mc) in enumerate(zip(milestones, m_labels, m_colors)):
        col = f"pct_repeat_{m}d"
        ax.bar(x + i*width, ret_merged[col]*100, width, label=ml, color=mc, edgecolor="white")
    ax.set_xticks(x + width)
    ax.set_xticklabels(TIER_ORDER)
    ax.set_ylabel("% Retained")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(PROPOSAL_DIR / "proposal_retention_milestones_by_tier.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: proposal_retention_milestones_by_tier.png")

In [59]:
# ── Customer lifespan (repeat buyers only) ────────────────────────────────────
print("CUSTOMER LIFESPAN — by Tier (repeat buyers only)")
print("(Days between first and last order — measures how long they stayed engaged)")
print()

life_rows = []
for tier in TIER_ORDER:
    g = df_active[
        (df_active["tier"] == tier) &
        (df_active["finals_orders"] >= 2) &
        (df_active["lifespan_days"].notna())
    ]["lifespan_days"]
    life_rows.append({
        "tier":             tier,
        "n_repeat_buyers":  len(g),
        "median_lifespan":  g.median() if len(g) else np.nan,
        "mean_lifespan":    g.mean()   if len(g) else np.nan,
        "p25":              g.quantile(0.25) if len(g) else np.nan,
        "p75":              g.quantile(0.75) if len(g) else np.nan,
    })

life_df = pd.DataFrame(life_rows)
print(life_df.to_string(index=False,
    formatters={c: lambda x: f"{x:.0f}d" if not pd.isna(x) else "—"
                for c in ["median_lifespan","mean_lifespan","p25","p75"]}))

CUSTOMER LIFESPAN — by Tier (repeat buyers only)
(Days between first and last order — measures how long they stayed engaged)

  tier  n_repeat_buyers median_lifespan mean_lifespan p25  p75
Tier 1              500            224d          329d 97d 457d
Tier 2              132             96d          184d 30d 240d
Tier 3              195             93d          181d 42d 268d
Tier 4               60             47d          116d 17d 130d


## 10. Budget Framework — Recommended Retention Spend

Budget principle: **spend a percentage of each tier's avg GP per customer per year.**

This ensures:
- Higher-value tiers receive proportionally more investment
- Every initiative is self-funding within the same customer's GP
- LP can adjust the `RETENTION_BUDGET_PCT` parameters in Section 0 and recalculate

We model the **total budget envelope** and the **expected GP return** under base scenario
assumptions for each initiative.

In [60]:
print("RETENTION BUDGET FRAMEWORK")
print(f"(Budget = RETENTION_BUDGET_PCT × avg GP/customer × n_customers)")
print()

budget_rows = []
for tier in TIER_ORDER:
    p   = prof_df[prof_df["tier"] == tier].iloc[0]
    pct = RETENTION_BUDGET_PCT[tier]
    budget_per_cust   = p["avg_gp"] * pct
    total_tier_budget = p["n_customers"] * budget_per_cust
    total_tier_gp     = p["total_gp"]
    budget_rows.append({
        "tier":               tier,
        "n_customers":        p["n_customers"],
        "avg_gp":             p["avg_gp"],
        "budget_pct":         pct,
        "budget_per_customer": budget_per_cust,
        "total_budget":       total_tier_budget,
        "total_gp":           total_tier_gp,
        "budget_as_pct_gp":   total_tier_budget / total_tier_gp if total_tier_gp > 0 else 0,
    })

budget_df = pd.DataFrame(budget_rows)

fmt_b = {
    "avg_gp":              lambda x: f"S${x:,.0f}",
    "budget_pct":          lambda x: f"{x:.0%}",
    "budget_per_customer": lambda x: f"S${x:,.0f}",
    "total_budget":        lambda x: f"S${x:,.0f}",
    "total_gp":            lambda x: f"S${x:,.0f}",
    "budget_as_pct_gp":    lambda x: f"{x:.1%}",
}

print(budget_df.to_string(index=False, formatters=fmt_b))
print()
total_budget_all = budget_df["total_budget"].sum()
total_gp_all     = budget_df["total_gp"].sum()
print(f"TOTAL retention budget (all tiers) : S${total_budget_all:,.0f}")
print(f"TOTAL GP protected (active pool)   : S${total_gp_all:,.0f}")
print(f"Budget as % of total GP            : {total_budget_all/total_gp_all:.1%}")

RETENTION BUDGET FRAMEWORK
(Budget = RETENTION_BUDGET_PCT × avg GP/customer × n_customers)

  tier  n_customers avg_gp budget_pct budget_per_customer total_budget  total_gp budget_as_pct_gp
Tier 1          500  S$265        15%                S$40     S$19,904 S$132,691            15.0%
Tier 2          358  S$181        12%                S$22      S$7,763  S$64,691            12.0%
Tier 3          195   S$83        12%                S$10      S$1,940  S$16,163            12.0%
Tier 4          663   S$81        10%                 S$8      S$5,355  S$53,548            10.0%

TOTAL retention budget (all tiers) : S$34,961
TOTAL GP protected (active pool)   : S$267,094
Budget as % of total GP            : 13.1%


In [61]:
print("ROI SCENARIOS — Expected GP Return per Initiative")
print("=" * 80)

# Pull pre-computed values from earlier cells
avg_gp_t1  = prof_df[prof_df["tier"] == "Tier 1"]["avg_gp"].values[0]
n_t1       = int(prof_df[prof_df["tier"] == "Tier 1"]["n_customers"].values[0])

avg_gp_t2  = prof_df[prof_df["tier"] == "Tier 2"]["avg_gp"].values[0]

avg_gp_t3  = prof_df[prof_df["tier"] == "Tier 3"]["avg_gp"].values[0]
n_t3       = int(prof_df[prof_df["tier"] == "Tier 3"]["n_customers"].values[0])
n_t3_nosub = n_tier3_total - int(n_subscribed)

n_t4       = int(prof_df[prof_df["tier"] == "Tier 4"]["n_customers"].values[0])

scenarios = [
    {
        "Initiative": "Tier 1 retention (protect active)",
        "Target customers": n_t1,
        "Cost (S$)": int(budget_df[budget_df["tier"] == "Tier 1"]["total_budget"].values[0]),
        "Base GP return (S$)": int(n_t1 * avg_gp_t1 * 0.10),  # 10% churn prevented
        "Notes": "10% churn prevention assumption",
    },
    {
        "Initiative": "Tier 1 + Tier 2 win-back (180-365d)",
        "Target customers": int(wb_df[wb_df["tier"].isin(["Tier 1", "Tier 2"])]["n_churned"].sum() * 0.5),
        "Cost (S$)": int(wb_df[wb_df["tier"].isin(["Tier 1", "Tier 2"])]["n_churned"].sum() * 0.5 * WINBACK_COST_PER_PERSON),
        "Base GP return (S$)": int(wb_df[wb_df["tier"].isin(["Tier 1", "Tier 2"])]["conv_10_half"].sum()),
        "Notes": "10% conv at 50% GP; 180-365d segment only",
    },
    {
        "Initiative": "Tier 3 win-back (180-365d)",
        "Target customers": int(rec_df[rec_df["tier"] == "Tier 3"]["n_churned"].values[0] * 0.5),
        "Cost (S$)": int(rec_df[rec_df["tier"] == "Tier 3"]["n_churned"].values[0] * 0.5 * WINBACK_COST_PER_PERSON),
        "Base GP return (S$)": int(wb_df[wb_df["tier"] == "Tier 3"]["conv_10_half"].values[0]),
        "Notes": "10% conv at 50% GP; 180-365d segment only",
    },
    {
        "Initiative": "Tier 3 subscription conversion",
        "Target customers": n_t3_nosub,
        "Cost (S$)": int(n_t3_nosub * 5),  # S$5/email + incentive
        "Base GP return (S$)": int(n_t3_nosub * 0.50 * avg_gp_t3),
        "Notes": "50% conversion; GP locked-in vs at-risk reorder",
    },
    {
        "Initiative": "Tier 4 2nd-purchase nudge",
        "Target customers": n_t4,
        "Cost (S$)": int(budget_df[budget_df["tier"] == "Tier 4"]["total_budget"].values[0]),
        "Base GP return (S$)": int(n_t4 * 0.10 * gp_uplift_per_customer),
        "Notes": "10% upgrade to Tier 2/3; GP uplift per customer",
    },
]

roi_df = pd.DataFrame(scenarios)
roi_df["Net GP (S$)"] = roi_df["Base GP return (S$)"] - roi_df["Cost (S$)"]
roi_df["ROI"]         = roi_df["Base GP return (S$)"] / roi_df["Cost (S$)"]

print(roi_df[["Initiative", "Target customers", "Cost (S$)",
              "Base GP return (S$)", "Net GP (S$)", "ROI", "Notes"]]
      .to_string(index=False,
          formatters={
              "Cost (S$)":           lambda x: f"S${x:,.0f}",
              "Base GP return (S$)": lambda x: f"S${x:,.0f}",
              "Net GP (S$)":         lambda x: f"S${x:,.0f}",
              "ROI":                 lambda x: f"{x:.1f}x",
          }))

print()
print(f"TOTAL cost across all initiatives : S${roi_df['Cost (S$)'].sum():,}")
print(f"TOTAL base GP return             : S${roi_df['Base GP return (S$)'].sum():,}")
print(f"TOTAL net GP (return minus cost) : S${roi_df['Net GP (S$)'].sum():,}")
print(f"Blended ROI                      : {roi_df['Base GP return (S$)'].sum() / roi_df['Cost (S$)'].sum():.1f}x")

ROI SCENARIOS — Expected GP Return per Initiative
                         Initiative  Target customers Cost (S$) Base GP return (S$) Net GP (S$)  ROI                                           Notes
  Tier 1 retention (protect active)               500  S$19,903            S$13,269    S$-6,634 0.7x                 10% churn prevention assumption
Tier 1 + Tier 2 win-back (180-365d)               333  S$10,005             S$7,354    S$-2,651 0.7x       10% conv at 50% GP; 180-365d segment only
         Tier 3 win-back (180-365d)                79   S$2,385               S$621    S$-1,764 0.3x       10% conv at 50% GP; 180-365d segment only
     Tier 3 subscription conversion                93     S$465             S$3,854     S$3,389 8.3x 50% conversion; GP locked-in vs at-risk reorder
          Tier 4 2nd-purchase nudge               663   S$5,354             S$4,338    S$-1,016 0.8x 10% upgrade to Tier 2/3; GP uplift per customer

TOTAL cost across all initiatives : S$38,112
TOTAL base

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Retention Budget — Cost vs Expected GP Return by Initiative",
             fontsize=13, fontweight="bold")

short_labels = [
    "Tier 1\nretention",
    "Tier 1+2\nwin-back",
    "Tier 3\nwin-back",
    "Tier 3\nsub conv",
    "Tier 4\nnudge",
]

ax = axes[0]
x = np.arange(len(roi_df))
width = 0.35
ax.bar(x - width/2, roi_df["Cost (S$)"],          width, label="Cost",      color="#E76F51", edgecolor="white")
ax.bar(x + width/2, roi_df["Base GP return (S$)"], width, label="GP Return", color="#2A9D8F", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=9)
ax.set_ylabel("SGD")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"S${x:,.0f}"))
ax.legend()
ax.set_title("Cost vs Expected GP Return")

ax = axes[1]
bar_colors_roi = ["#2A9D8F" if r >= 1 else "#E76F51" for r in roi_df["ROI"]]
bars = ax.bar(short_labels, roi_df["ROI"], color=bar_colors_roi, edgecolor="white")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="Break-even")
ax.set_ylabel("ROI (x)")
ax.set_title("ROI per Initiative")
ax.legend(fontsize=9)
for bar, val in zip(bars, roi_df["ROI"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.1f}x", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(PROPOSAL_DIR / "proposal_budget_roi.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: proposal_budget_roi.png")

## 11. Business Case Summary

Full derivation of every number cited in the CEO retention strategy proposal.

In [ ]:
print("=" * 70)
print("LUSHPROTEIN — CEO RETENTION STRATEGY: BUSINESS CASE SUMMARY")
print("=" * 70)
print(f"Data end date : {REFERENCE_DATE.date()}")
print(f"Total pool    : 4,290 customers (finals-eligible, excl. marketplace)")
print()

print("TIER COMPOSITION (D1 + D2 profit only)")
print("-" * 70)
for _, row in prof_df.iterrows():
    print(f"  {row['tier']:<10} {row['n_customers']:>5,} customers | "
          f"Avg GP S${row['avg_gp']:>7,.0f} | "
          f"Avg orders {row['avg_orders']:.1f} | "
          f"{row['pct_subscribed']:.0%} subscribed | "
          f"{row['pct_of_active_gp']:.1%} of active GP")
print()

print("CHURN REALITY (biggest risk)")
print("-" * 70)
for _, row in rec_df.iterrows():
    print(f"  {row['tier']:<10} {row['pct_churned']:.0%} likely churned "
          f"({row['n_churned']:.0f} customers) | "
          f"{row['pct_active']:.0%} active ({row['n_active']:.0f})")
print()

print("KEY BEHAVIOURAL SIGNALS")
print("-" * 70)
for tier in TIER_ORDER:
    t2   = t2nd_df[t2nd_df["tier"] == tier]
    life = life_df[life_df["tier"] == tier]
    med_2nd  = t2["median_days"].values[0]   if len(t2)   > 0 else np.nan
    med_life = life["median_lifespan"].values[0] if len(life) > 0 else np.nan
    print(f"  {tier:<10} Median 2nd purchase: {med_2nd:.0f}d | "
          f"Median lifespan (repeat buyers): {med_life:.0f}d")
print()

print("RETENTION INITIATIVES SUMMARY")
print("-" * 70)
for _, row in roi_df.iterrows():
    print(f"  {row['Initiative']:<42} "
          f"Cost S${row['Cost (S$)']:>7,.0f} | "
          f"Return S${row['Base GP return (S$)']:>7,.0f} | "
          f"ROI {row['ROI']:.1f}x")
print("-" * 70)
print(f"  {'TOTAL':<42} "
      f"Cost S${roi_df['Cost (S$)'].sum():>7,.0f} | "
      f"Return S${roi_df['Base GP return (S$)'].sum():>7,.0f} | "
      f"ROI {roi_df['Base GP return (S$)'].sum()/roi_df['Cost (S$)'].sum():.1f}x")
print()

print(f"OUTPUTS → {PROPOSAL_DIR}")
print("-" * 70)
outputs = [
    "tier_customer_map.csv                      — Customer ID → tier + all financial fields",
    "proposal_tier_profiles.png                 — Tier profiles chart",
    "proposal_recency_by_tier.png               — Churn/recency breakdown",
    "proposal_repeat_funnel_by_tier.png         — Repeat purchase funnel",
    "proposal_time_to_2nd_by_tier.png           — 2nd purchase intervention window",
    "proposal_retention_milestones_by_tier.png  — 3/6/12 month retention",
    "proposal_budget_roi.png                    — Cost vs GP return per initiative",
]
for o in outputs:
    print(f"  {o}")
print("=" * 70)